# 01.04 — Train, Validate, Test

**Module:** 01 — ML Fundamentals  
**Prerequisites:** 01.01–01.03  
**Difficulty:** introductory / foundational  
**Estimated time:** 45–60 minutes

This lesson uses synthetic, deterministic, dimensionless values. It asks: **why do we separate training, validation, and test data, and what can each split legitimately tell us?**

**Outcomes:** distinguish split roles; compute split-specific MSE; select a candidate with validation; reserve test for a final report; identify leakage; and state why one fixed split is not a generalisation guarantee.


## 1. The problem

A score can look useful while answering the wrong question. Training data may be used to fit candidates, validation data may guide development choices, and test data should be reserved for a final check after those choices are frozen. This lesson studies information roles, not a fitting algorithm.

## 2. Toy dataset and fixed partitions

The synthetic signal is `y = 2x + 1`. The fixed, disjoint partitions are:

| split | x | y |
|---|---|---|
| train | `[1, 2, 3, 4]` | `[3, 5, 7, 9]` |
| validation | `[5, 6, 7]` | `[11, 13, 15]` |
| test | `[8, 9, 10, 11, 12]` | `[17, 19, 21, 23, 25]` |

Two fixed prediction tables are supplied; no estimator or random split is used. Candidate A predicts every target exactly. Candidate B predicts `[10, 12.5, 15.5]` on validation and `[16, 18.5, 21.5, 24.5, 28]` on test.

In [ ]:
import numpy as np

x_train = np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float64)
y_train = np.array([3.0, 5.0, 7.0, 9.0], dtype=np.float64)
x_validation = np.array([5.0, 6.0, 7.0], dtype=np.float64)
y_validation = np.array([11.0, 13.0, 15.0], dtype=np.float64)
x_test = np.array([8.0, 9.0, 10.0, 11.0, 12.0], dtype=np.float64)
y_test = np.array([17.0, 19.0, 21.0, 23.0, 25.0], dtype=np.float64)

predictions = {
    "A": (y_train.copy(), y_validation.copy(), y_test.copy()),
    "B": (
        np.array([3.0, 5.0, 7.0, 9.0], dtype=np.float64),
        np.array([10.0, 12.5, 15.5], dtype=np.float64),
        np.array([16.0, 18.5, 21.5, 24.5, 28.0], dtype=np.float64),
    ),
}
targets_by_split = (y_train, y_validation, y_test)


## 3. Intuition before math

`train → fit candidates → validation → choose candidate → test → report once`

Held out means excluded from the stated fitting or selection step; it does not mean magically representative of every future dataset. The test split must not feed back into candidate selection.

## 4. Mathematical definition

For split `S`, residual `e_i = y_i − ŷ_i` and
`MSE_S = (1/|S|) Σ e_i²`. We compare candidates using validation MSE, then evaluate the selected candidate on test.

## 5. Manual walkthrough

| candidate | train MSE | validation MSE | test MSE |
|---|---:|---:|---:|
| A | `0.0` | `0.0` | `0.0` |
| B | `0.0` | `0.5` | `2.55` |

For B on validation, squared losses are `[1, 0.25, 0.25]`, so MSE is `1.5/3 = 0.5`. On test they are `[1, 0.25, 0.25, 2.25, 9]`, so MSE is `12.75/5 = 2.55`. Validation selects A because `0.0 < 0.5`; test is reported only after that choice is frozen.

In [ ]:
def mean_squared_error(targets, predictions):
    if targets.dtype != np.float64 or predictions.dtype != np.float64:
        raise ValueError("arrays must use float64")
    if targets.ndim != 1 or predictions.ndim != 1 or targets.size == 0:
        raise ValueError("arrays must be non-empty 1-D arrays")
    if targets.shape != predictions.shape:
        raise ValueError("arrays must have matching shapes")
    return np.mean((targets - predictions) ** 2)

mse = {
    name: tuple(mean_squared_error(targets, prediction)
               for targets, prediction in zip(targets_by_split, values, strict=True))
    for name, values in predictions.items()
}
mse


In [ ]:
np.testing.assert_allclose(mse["A"], [0.0, 0.0, 0.0], rtol=0.0, atol=1e-12)
np.testing.assert_allclose(mse["B"], [0.0, 0.5, 2.55], rtol=0.0, atol=1e-12)
selected = min(mse, key=lambda name: mse[name][1])
assert selected == "A"
print(f"Selected on validation: {selected}; test MSE: {mse[selected][2]:.2f}")


## 6. Independent verification

The companion verifier recomputes the frozen values with a separate implementation and checks the literal test arithmetic. It does not import this notebook or the private implementation.

## 7. Visual explanation

![Train, validation, and test information roles](train_validate_test_visual.png)

*Figure 1.* Training may fit candidates, validation guides selection, and test is reported once after selection is frozen. The values are synthetic and dimensionless; one fixed split is not a generalisation guarantee.

**Alt text:** Static two-panel figure. The left panel shows train leading to candidate parameters, validation leading to a selection signal, and test leading to a final held-out summary, with a warning not to feed test rows back into selection. The right panel shows hatched bars: candidate A has MSE `0.00` on all splits; candidate B has `0.00`, `0.50`, and `2.55` on train, validation, and test. A callout says A is selected on validation before test is reported.

## 8. Failure modes and exercises

**Failure modes:** selecting by test, repeatedly reusing test results, overlapping rows, and preparing inputs with statistics from held-out rows all leak information. A held-out score is not a universal quality judgment.

1. Which split fits candidates, which guides selection, and which is reported last?
2. Reproduce B's validation MSE and test MSE from the squared losses above.
3. Why is selecting B because of a lower test score test peeking?
4. Why does A's test MSE of `0.0` not prove perfect future performance?

Answers: train, validation, test; `0.5` and `2.55`; test would become a selection signal; the example is a fixed synthetic partition, not a generalisation guarantee.

## 9. References and continuity

James, Witten, Hastie, and Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Chapter 5; Goodfellow, Bengio, and Courville, *Deep Learning*, §5.6; Hastie, Tibshirani, and Friedman, *The Elements of Statistical Learning*, Chapter 7; and the NumPy 1.26 documentation. The scikit-learn model-selection guide is a conceptual pointer only. Random splitting, cross-validation, preprocessing APIs, estimators, and deployment monitoring are outside this lesson.